# 10 - Best pre-window CNN: 601_dilated_rcaug_rctta

Notebook này đóng gói phiên bản tốt nhất trước khi thử `window attention`:

- Window: `601 bp`
- Input: `ref + alt + mutation marker` 9-channel
- Positional encoding: sinusoidal, `8` channel
- Model: dilated 1D CNN
- Split gốc: gene-group split
- Augmentation: reverse-complement augment khi train
- Evaluation: reverse-complement TTA
- Hard-negative mining: 2 epoch

Kết quả lịch sử của bản này:

- Test ROC-AUC: `0.9557`
- Test PR-AUC: `0.8431`
- F1 @ 0.5: `0.7366`
- Test F1 tại threshold chọn từ validation F1: `0.7485`

Ghi chú: bản `dilated_window_attention` sau đó nhỉnh hơn, nhưng notebook này giữ đúng mốc baseline mạnh nhất trước attention.

## Latest local full run

Đã chạy lại full trên dataset `fixed_refseq` với `chromosome` split ngày 2026-05-18. Kết quả chính được lưu trong `processed/cnn_gene_group_positional_encoding_601_fixed_refseq_dilated_rcaug_rctta_chromosome_split_metrics.json`:

- Best val PR-AUC: `0.9200`
- Test ROC-AUC: `0.9755`
- Test PR-AUC: `0.8877`
- Test F1 @ 0.5: `0.7674`
- Test F1 tại best validation-F1 threshold: `0.7899`

Notebook vẫn để `RUN_TRAIN=False` mặc định để tránh vô tình ghi đè artifact.

## 1. Config

Mặc định `RUN_TRAIN = False` để tránh train lại và ghi đè checkpoint cũ. Nếu muốn train lại trên dataset đã fix RefSeq `.11/.12`, đổi `DATASET_TAG = "fixed_refseq"` và cân nhắc `SPLIT_MODE = "chromosome"`.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
PROJECT_DIR = PROJECT_DIR.resolve()

CONFIG = {
    "length": 601,
    "dataset_tag": "fixed_refseq",  # "" = artifact gốc; "fixed_refseq" = dataset đã sửa mapping GRCh38 RefSeq.
    "label_mode": "all",  # all = giữ Likely; definitive_only = chỉ Benign/Pathogenic, bỏ Likely.
    "split_mode": "genome_block",  # gene_group | purged_gene_group | chromosome | genome_block | time_only | space_time_block | space_only_matched
    "purge_distance_bp": 5000,
    "genome_block_size_bp": 1_000_000,
    "sequence_purge_mode": "exact_ref",  # none | exact_ref | exact_tensor
    "temporal_train_max_year": 2022,
    "temporal_val_year": 2023,
    "temporal_test_min_year": 2024,
    "drop_missing_last_evaluated": True,
    "exact_variant_purge": True,
    "match_space_only_train_size": True,
    "space_only_target_train_rows": None,
    "architecture": "dilated",
    "positional_encoding": "sinusoidal",
    "positional_dim": 8,
    "batch_size": 256,
    "epochs": 5,
    "hard_negative_epochs": 2,
    "learning_rate": 1e-3,
    "hard_negative_lr": 3e-4,
    "weight_decay": 1e-4,
    "scheduler": "none",      # giữ gần nhất với run tốt cũ trước khi thêm scheduler.
    "warmup_epochs": 0,
    "min_lr_ratio": 0.05,
    "grad_clip_norm": 0.0,    # 0/None nghĩa là không clip trong script hiện tại.
    "random_state": 42,
    "num_workers": 0,
    "imbalance_strategy": "weighted_sampler",  # none | weighted_sampler | pos_weight | legacy_sqrt_sampler_pos_weight
    "rc_augment": True,
    "rc_tta": True,
    "debug_max_train_rows": None,
    "debug_max_val_rows": None,
    "debug_max_test_rows": None,
}

RUN_TRAIN = True
FORCE_OVERWRITE = True

PROCESSED_DIR = PROJECT_DIR / "processed"
MODEL_DIR = PROJECT_DIR / "models"

HISTORICAL_METRICS = {
    "model_name": "cnn_gene_group_positional_encoding_601_dilated_rcaug_rctta",
    "test_roc_auc": 0.9556918925,
    "test_pr_auc": 0.8430617465,
    "test_accuracy": 0.9153853044,
    "test_precision_at_05": 0.6693132866,
    "test_recall_at_05": 0.8188488168,
    "test_f1_at_05": 0.7365681353,
    "best_val_f1_threshold": 0.6226215363,
    "test_f1_at_best_val_f1_threshold": 0.7485127665,
    "runtime_minutes": 15.883,
}

print("PROJECT_DIR:", PROJECT_DIR)


## Label mode

`CONFIG["label_mode"]` điều khiển cách dùng nhãn ClinVar:

- `"all"`: giữ cả `Benign`, `Likely benign`, `Pathogenic`, `Likely pathogenic`, và các nhãn gộp đã được dataset encode. Đây là cấu hình tương thích với kết quả cũ.
- `"definitive_only"`: chỉ train/evaluate trên các dòng có `ClinicalSignificance` đúng bằng `Benign` hoặc `Pathogenic`, bỏ các dòng `Likely ...` và nhãn gộp. Artifact sẽ có suffix `_definitive_only`.


## 2. Artifact names

Cell này tạo đúng tên file theo logic của training script. Nếu `dataset_tag=""`, tên model chính là `cnn_gene_group_positional_encoding_601_dilated_rcaug_rctta`.

In [ ]:
def build_safe_name(config):
    dataset_suffix = str(config["length"])
    if config["dataset_tag"]:
        dataset_suffix = f"{dataset_suffix}_{config['dataset_tag']}"

    positional_name = {
        "none": "no_positional_encoding",
        "sinusoidal": "positional_encoding",
        "relative": "relative_positional_encoding",
    }[config["positional_encoding"]]

    safe_name = f"cnn_gene_group_{positional_name}_{dataset_suffix}"
    if config["architecture"] != "baseline":
        safe_name = f"{safe_name}_{config['architecture']}"
    if config["rc_augment"]:
        safe_name = f"{safe_name}_rcaug"
    if config["rc_tta"]:
        safe_name = f"{safe_name}_rctta"
    if config["split_mode"] == "purged_gene_group":
        safe_name = f"{safe_name}_purged{config['purge_distance_bp']}"
    if config["split_mode"] == "chromosome":
        safe_name = f"{safe_name}_chromosome_split"
    if config["split_mode"] == "genome_block":
        safe_name = f"{safe_name}_genome_block{config['genome_block_size_bp']}_purged{config['purge_distance_bp']}"
    if config["split_mode"] == "time_only":
        safe_name = f"{safe_name}_time_trainlte{config['temporal_train_max_year']}_val{config['temporal_val_year']}_testgte{config['temporal_test_min_year']}_purged{config['purge_distance_bp']}"
    if config["split_mode"] == "space_time_block":
        safe_name = f"{safe_name}_space_time_block{config['genome_block_size_bp']}_trainlte{config['temporal_train_max_year']}_val{config['temporal_val_year']}_testgte{config['temporal_test_min_year']}_purged{config['purge_distance_bp']}"
    if config["split_mode"] == "space_only_matched":
        safe_name = f"{safe_name}_space_only_matched_block{config['genome_block_size_bp']}_trainmatch_trainlte{config['temporal_train_max_year']}_purged{config['purge_distance_bp']}"
    if config["sequence_purge_mode"] != "none":
        safe_name = f"{safe_name}_seqpurge_{config['sequence_purge_mode']}"
    if config.get("label_mode", "all") != "all":
        safe_name = f"{safe_name}_{config['label_mode']}"
    if config.get("imbalance_strategy", "legacy_sqrt_sampler_pos_weight") != "legacy_sqrt_sampler_pos_weight":
        safe_name = f"{safe_name}_imbalance_{config['imbalance_strategy']}"
    if config["random_state"] != 42:
        safe_name = f"{safe_name}_seed{config['random_state']}"
    return dataset_suffix, safe_name

DATASET_SUFFIX, SAFE_NAME = build_safe_name(CONFIG)
paths = {
    "x": PROCESSED_DIR / f"X_ref_alt_marker_{DATASET_SUFFIX}.npy",
    "y": PROCESSED_DIR / f"y_{DATASET_SUFFIX}.npy",
    "metadata": PROCESSED_DIR / f"clinvar_training_metadata_{DATASET_SUFFIX}.parquet",
    "model": MODEL_DIR / f"clinvar_{SAFE_NAME}_pytorch.pt",
    "metrics": PROCESSED_DIR / f"{SAFE_NAME}_metrics.json",
    "history": PROCESSED_DIR / f"{SAFE_NAME}_training_history_pytorch.parquet",
    "predictions": PROCESSED_DIR / f"{SAFE_NAME}_test_predictions_pytorch.parquet",
    "thresholds": PROCESSED_DIR / f"{SAFE_NAME}_threshold_table_pytorch.parquet",
    "val_thresholds": PROCESSED_DIR / f"{SAFE_NAME}_val_threshold_table_pytorch.parquet",
    "split_indices": PROCESSED_DIR / f"{SAFE_NAME}_split_indices.npz",
}

print("DATASET_SUFFIX:", DATASET_SUFFIX)
print("SAFE_NAME:", SAFE_NAME)
for key, value in paths.items():
    print(f"{key:12s}", value, "exists=" + str(value.exists()))

## 3. Load dataset và chọn nhãn

Notebook này không gọi script train nữa. Các bước train/eval bên dưới được gộp trực tiếp vào notebook.

In [ ]:
import hashlib
import random
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from IPython.display import display


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def select_label_indices(meta, label_mode):
    if label_mode == "all":
        return np.arange(len(meta), dtype=np.int64)
    if label_mode == "definitive_only":
        clinical = meta["ClinicalSignificance"].fillna("").astype(str).str.strip()
        return np.flatnonzero(clinical.isin(["Benign", "Pathogenic"]).to_numpy()).astype(np.int64)
    raise ValueError(f"Unsupported label_mode: {label_mode}")

set_seed(CONFIG["random_state"])

x = np.load(paths["x"], mmap_mode="r")
y = np.load(paths["y"], mmap_mode="r")
metadata = pd.read_parquet(paths["metadata"])
active_idx = select_label_indices(metadata, CONFIG["label_mode"])
metadata_for_split = metadata.iloc[active_idx].reset_index(drop=True)
y_for_split = np.asarray(y[active_idx], dtype=np.int8)

print("x:", x.shape, x.dtype)
print("y:", y.shape, y.dtype)
print("metadata:", metadata.shape)
print("LABEL_MODE:", CONFIG["label_mode"])
print("active rows:", f"{len(active_idx):,}", "/", f"{len(metadata):,}")
print("active labels:", dict(zip(*np.unique(y_for_split, return_counts=True))))

## 4. Split strict: genome block + purge + sequence purge

Notebook này hỗ trợ thêm pseudo-temporal split dựa trên `LastEvaluated`: `time_only`, `space_time_block`, và `space_only_matched`. Với các mode temporal, `LastEvaluated` phải có trong metadata CNN.


In [ ]:
def make_groups(meta):
    groups = meta["GeneSymbol"].fillna("unknown").astype(str).to_numpy(copy=True)
    unknown = (groups == "") | (groups == "unknown") | (groups == "nan")
    row_ids = np.arange(len(groups)).astype(str)
    groups[unknown] = np.char.add("unknown_row_", row_ids[unknown])
    return groups


def make_chromosome_groups(meta):
    groups = meta["Chromosome"].fillna("unknown").astype(str).str.replace("chr", "", case=False, regex=False).to_numpy(copy=True)
    unknown = (groups == "") | (groups == "unknown") | (groups == "nan")
    row_ids = np.arange(len(groups)).astype(str)
    groups[unknown] = np.char.add("unknown_chr_row_", row_ids[unknown])
    return groups


def make_genome_block_groups(meta, chromosome_groups, block_size_bp):
    positions = pd.to_numeric(meta["PositionVCF"], errors="coerce")
    blocks = ((positions - 1) // block_size_bp).astype("Int64")
    chrom = pd.Series(chromosome_groups, index=meta.index).astype(str)
    block_groups = chrom + ":block_" + blocks.astype(str)
    invalid = positions.isna() | chrom.isin(["", "unknown", "nan"]) | blocks.isna()
    if invalid.any():
        row_ids = pd.Series(np.arange(len(meta)), index=meta.index).astype(str)
        block_groups.loc[invalid] = "unknown_block_row_" + row_ids.loc[invalid]
    return block_groups.to_numpy(dtype=str, copy=True)


def make_group_split(labels, groups, random_state):
    all_idx = np.arange(len(labels))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=random_state)
    train_idx, temp_idx = next(gss1.split(all_idx, labels, groups=groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=random_state)
    val_rel, test_rel = next(gss2.split(temp_idx, labels[temp_idx], groups=groups[temp_idx]))
    return train_idx, temp_idx[val_rel], temp_idx[test_rel]


def maybe_subsample(indices, max_rows, labels, seed):
    if max_rows is None or len(indices) <= max_rows:
        return indices
    rng = np.random.default_rng(seed)
    selected = []
    for label in [0, 1]:
        label_idx = indices[labels[indices] == label]
        n_label = int(round(max_rows * len(label_idx) / len(indices)))
        n_label = min(len(label_idx), max(1, n_label))
        selected.append(rng.choice(label_idx, size=n_label, replace=False))
    out = np.concatenate(selected)
    rng.shuffle(out)
    return out


def nearest_distance_to_reference(meta, query_idx, reference_idx):
    ref_pos_by_chr = {}
    for chrom, sub in meta.iloc[reference_idx].groupby("Chromosome", sort=False):
        positions = pd.to_numeric(sub["PositionVCF"], errors="coerce").dropna()
        ref_pos_by_chr[str(chrom)] = np.sort(positions.to_numpy(dtype=np.int64))
    query = meta.iloc[query_idx]
    distances = np.full(len(query_idx), np.inf, dtype=np.float64)
    for i, (chrom, pos) in enumerate(zip(query["Chromosome"].astype(str), pd.to_numeric(query["PositionVCF"], errors="coerce"))):
        if pd.isna(pos):
            continue
        ref_positions = ref_pos_by_chr.get(str(chrom))
        if ref_positions is None or len(ref_positions) == 0:
            continue
        pos_int = int(pos)
        insert_at = np.searchsorted(ref_positions, pos_int)
        best = np.inf
        if insert_at > 0:
            best = min(best, abs(pos_int - int(ref_positions[insert_at - 1])))
        if insert_at < len(ref_positions):
            best = min(best, abs(pos_int - int(ref_positions[insert_at])))
        distances[i] = best
    return distances


def distance_summary(distances):
    finite = distances[np.isfinite(distances)]
    if len(finite) == 0:
        return {"min": float("inf"), "p5": float("inf"), "median": float("inf"), "p95": float("inf")}
    return {"min": float(np.min(finite)), "p5": float(np.percentile(finite, 5)), "median": float(np.median(finite)), "p95": float(np.percentile(finite, 95))}


def purge_nearby_splits(meta, train_idx, val_idx, test_idx, purge_distance_bp):
    val_dist = nearest_distance_to_reference(meta, val_idx, train_idx)
    keep_val = val_dist >= purge_distance_bp
    kept_val_idx = val_idx[keep_val]
    test_ref = np.concatenate([train_idx, kept_val_idx])
    test_dist = nearest_distance_to_reference(meta, test_idx, test_ref)
    keep_test = test_dist >= purge_distance_bp
    kept_test_idx = test_idx[keep_test]
    audit = {
        "val_rows_before_purge": int(len(val_idx)),
        "val_rows_after_purge": int(len(kept_val_idx)),
        "val_rows_purged": int((~keep_val).sum()),
        "test_rows_before_purge": int(len(test_idx)),
        "test_rows_after_purge": int(len(kept_test_idx)),
        "test_rows_purged": int((~keep_test).sum()),
        "val_nearest_train_distance_bp": distance_summary(nearest_distance_to_reference(meta, kept_val_idx, train_idx)),
        "test_nearest_train_val_distance_bp": distance_summary(nearest_distance_to_reference(meta, kept_test_idx, test_ref)),
    }
    return kept_val_idx, kept_test_idx, val_idx[~keep_val], test_idx[~keep_test], audit


def variant_keys(meta, indices):
    sub = meta.iloc[indices]
    return (
        sub["Chromosome"].astype(str)
        + ":" + pd.to_numeric(sub["PositionVCF"], errors="coerce").astype("Int64").astype(str)
        + ":" + sub["ReferenceAlleleVCF"].astype(str)
        + ":" + sub["AlternateAlleleVCF"].astype(str)
    ).to_numpy(dtype=str)


def purge_exact_variant_splits(meta, train_idx, val_idx, test_idx):
    train_keys = set(variant_keys(meta, train_idx))
    val_keys = variant_keys(meta, val_idx)
    keep_val = np.array([key not in train_keys for key in val_keys], dtype=bool)
    kept_val_idx = val_idx[keep_val]
    reference_keys = train_keys | set(variant_keys(meta, kept_val_idx))
    test_keys = variant_keys(meta, test_idx)
    keep_test = np.array([key not in reference_keys for key in test_keys], dtype=bool)
    kept_test_idx = test_idx[keep_test]
    audit = {
        "enabled": True,
        "val_rows_before_exact_variant_purge": int(len(val_idx)),
        "val_rows_after_exact_variant_purge": int(len(kept_val_idx)),
        "val_rows_purged_by_exact_variant": int((~keep_val).sum()),
        "test_rows_before_exact_variant_purge": int(len(test_idx)),
        "test_rows_after_exact_variant_purge": int(len(kept_test_idx)),
        "test_rows_purged_by_exact_variant": int((~keep_test).sum()),
    }
    return kept_val_idx, kept_test_idx, val_idx[~keep_val], test_idx[~keep_test], audit


def context_hash(global_idx, mode):
    if mode == "exact_ref":
        arr = np.ascontiguousarray(x[int(global_idx), :, :4])
    elif mode == "exact_tensor":
        arr = np.ascontiguousarray(x[int(global_idx)])
    else:
        raise ValueError(f"Unsupported sequence purge mode: {mode}")
    return hashlib.blake2b(arr.view(np.uint8), digest_size=16).digest()


def purge_sequence_context_splits(train_global_idx, val_global_idx, test_global_idx, mode):
    if mode == "none":
        return val_global_idx, test_global_idx, np.asarray([], dtype=np.int64), np.asarray([], dtype=np.int64), {"sequence_purge_mode": mode}
    train_hashes = {context_hash(i, mode) for i in tqdm(train_global_idx, desc="hash train contexts", leave=False)}
    keep_val = np.array([context_hash(i, mode) not in train_hashes for i in tqdm(val_global_idx, desc="purge val sequence duplicates", leave=False)])
    kept_val_global = val_global_idx[keep_val]
    val_hashes = {context_hash(i, mode) for i in tqdm(kept_val_global, desc="hash kept val contexts", leave=False)}
    reference_hashes = train_hashes | val_hashes
    keep_test = np.array([context_hash(i, mode) not in reference_hashes for i in tqdm(test_global_idx, desc="purge test sequence duplicates", leave=False)])
    kept_test_global = test_global_idx[keep_test]
    audit = {
        "sequence_purge_mode": mode,
        "val_rows_before_sequence_purge": int(len(val_global_idx)),
        "val_rows_after_sequence_purge": int(len(kept_val_global)),
        "val_rows_purged_by_sequence": int((~keep_val).sum()),
        "test_rows_before_sequence_purge": int(len(test_global_idx)),
        "test_rows_after_sequence_purge": int(len(kept_test_global)),
        "test_rows_purged_by_sequence": int((~keep_test).sum()),
    }
    return kept_val_global, kept_test_global, val_global_idx[~keep_val], test_global_idx[~keep_test], audit


def label_count_dict(labels, indices):
    label_values, counts = np.unique(labels[indices], return_counts=True)
    return {int(k): int(v) for k, v in zip(label_values, counts)}


def prevalence(labels, indices):
    if len(indices) == 0:
        return float("nan")
    return float(np.mean(labels[indices].astype(np.float64)))


def year_summary(years, indices):
    if years is None or len(indices) == 0:
        return {"min": None, "max": None, "counts": {}}
    values = years.iloc[indices].dropna().astype(int)
    if values.empty:
        return {"min": None, "max": None, "counts": {}}
    return {
        "min": int(values.min()),
        "max": int(values.max()),
        "counts": {int(k): int(v) for k, v in values.value_counts().sort_index().items()},
    }


def parse_last_evaluated_years(meta, config):
    if "LastEvaluated" not in meta.columns:
        raise ValueError(
            "Temporal split requires LastEvaluated. Rebuild fixed_refseq artifacts with LastEvaluated included."
        )
    parsed = pd.to_datetime(meta["LastEvaluated"], errors="coerce")
    years = parsed.dt.year
    missing = int(years.isna().sum())
    if missing and not config["drop_missing_last_evaluated"]:
        raise ValueError(f"LastEvaluated has {missing:,} missing/invalid rows. Set drop_missing_last_evaluated=True or fix metadata.")
    return years, missing


def apply_temporal_active_filter(active_idx, meta_for_split, labels, config):
    temporal_modes = {"time_only", "space_time_block", "space_only_matched"}
    if config["split_mode"] not in temporal_modes:
        return active_idx, meta_for_split, labels, None, 0
    years, missing = parse_last_evaluated_years(meta_for_split, config)
    keep = years.notna().to_numpy()
    return active_idx[keep], meta_for_split.loc[keep].reset_index(drop=True), labels[keep], years.loc[keep].reset_index(drop=True), missing


def split_by_time(years, config):
    all_idx = np.arange(len(years), dtype=np.int64)
    train_idx = all_idx[years.to_numpy() <= config["temporal_train_max_year"]]
    val_idx = all_idx[years.to_numpy() == config["temporal_val_year"]]
    test_idx = all_idx[years.to_numpy() >= config["temporal_test_min_year"]]
    return train_idx, val_idx, test_idx


def split_by_space_time(labels, genome_block_groups, years, config):
    block_train, block_val, block_test = make_group_split(labels, genome_block_groups, config["random_state"])
    train_blocks = set(genome_block_groups[block_train])
    val_blocks = set(genome_block_groups[block_val])
    test_blocks = set(genome_block_groups[block_test])
    year_values = years.to_numpy()
    all_idx = np.arange(len(labels), dtype=np.int64)
    train_idx = all_idx[np.isin(genome_block_groups, list(train_blocks)) & (year_values <= config["temporal_train_max_year"])]
    val_idx = all_idx[np.isin(genome_block_groups, list(val_blocks)) & (year_values == config["temporal_val_year"])]
    test_idx = all_idx[np.isin(genome_block_groups, list(test_blocks)) & (year_values >= config["temporal_test_min_year"])]
    return train_idx, val_idx, test_idx


def matched_space_only_target_rows(labels, genome_block_groups, years, config):
    if config.get("space_only_target_train_rows") is not None:
        return int(config["space_only_target_train_rows"])
    if not config.get("match_space_only_train_size", True):
        return None
    temporal_train, _, _ = split_by_space_time(labels, genome_block_groups, years, config)
    return int(len(temporal_train))


active_idx, metadata_for_split, y_for_split, temporal_years, dropped_missing_last_evaluated_rows = apply_temporal_active_filter(
    active_idx, metadata_for_split, y_for_split, CONFIG
)
if temporal_years is not None:
    print("Temporal active rows:", f"{len(active_idx):,}")
    print("Dropped missing LastEvaluated rows:", f"{dropped_missing_last_evaluated_rows:,}")

chromosome_groups = make_chromosome_groups(metadata_for_split)
genome_block_groups = make_genome_block_groups(metadata_for_split, chromosome_groups, CONFIG["genome_block_size_bp"])
gene_groups = make_groups(metadata_for_split)

time_only_modes = {"time_only"}
space_time_modes = {"space_time_block"}
space_only_matched_modes = {"space_only_matched"}
coordinate_purge_modes = {"purged_gene_group", "genome_block", "time_only", "space_time_block", "space_only_matched"}
block_disjoint_modes = {"genome_block", "space_time_block", "space_only_matched"}

if CONFIG["split_mode"] == "time_only":
    train_local, val_local, test_local = split_by_time(temporal_years, CONFIG)
elif CONFIG["split_mode"] == "space_time_block":
    train_local, val_local, test_local = split_by_space_time(y_for_split, genome_block_groups, temporal_years, CONFIG)
elif CONFIG["split_mode"] == "space_only_matched":
    train_local, val_local, test_local = make_group_split(y_for_split, genome_block_groups, CONFIG["random_state"])
    target_rows = matched_space_only_target_rows(y_for_split, genome_block_groups, temporal_years, CONFIG)
    if target_rows is not None:
        train_local = maybe_subsample(train_local, target_rows, y_for_split, CONFIG["random_state"] + 17)
else:
    if CONFIG["split_mode"] == "chromosome":
        split_groups = chromosome_groups
    elif CONFIG["split_mode"] == "genome_block":
        split_groups = genome_block_groups
    else:
        split_groups = gene_groups
    train_local, val_local, test_local = make_group_split(y_for_split, split_groups, CONFIG["random_state"])

if min(len(train_local), len(val_local), len(test_local)) == 0:
    raise ValueError(
        f"Split produced an empty partition: train={len(train_local)}, val={len(val_local)}, test={len(test_local)}. "
        "Adjust temporal years or split mode."
    )

exact_variant_purged_val_local = np.asarray([], dtype=np.int64)
exact_variant_purged_test_local = np.asarray([], dtype=np.int64)
exact_variant_purge_audit = {"enabled": bool(CONFIG.get("exact_variant_purge", False))}
if CONFIG.get("exact_variant_purge", False):
    val_local, test_local, exact_variant_purged_val_local, exact_variant_purged_test_local, exact_variant_purge_audit = purge_exact_variant_splits(
        metadata_for_split, train_local, val_local, test_local
    )

purged_val_local = np.asarray([], dtype=np.int64)
purged_test_local = np.asarray([], dtype=np.int64)
purge_audit = {}
if CONFIG["split_mode"] in coordinate_purge_modes:
    val_local, test_local, purged_val_local, purged_test_local, purge_audit = purge_nearby_splits(
        metadata_for_split, train_local, val_local, test_local, CONFIG["purge_distance_bp"]
    )

train_local = maybe_subsample(train_local, CONFIG["debug_max_train_rows"], y_for_split, CONFIG["random_state"])
val_local = maybe_subsample(val_local, CONFIG["debug_max_val_rows"], y_for_split, CONFIG["random_state"] + 1)
test_local = maybe_subsample(test_local, CONFIG["debug_max_test_rows"], y_for_split, CONFIG["random_state"] + 2)

sequence_purge_audit = {"sequence_purge_mode": CONFIG["sequence_purge_mode"]}
sequence_purged_val_idx = np.asarray([], dtype=np.int64)
sequence_purged_test_idx = np.asarray([], dtype=np.int64)
if CONFIG["sequence_purge_mode"] != "none":
    train_global_for_purge = active_idx[train_local]
    val_global_for_purge = active_idx[val_local]
    test_global_for_purge = active_idx[test_local]
    kept_val_global, kept_test_global, sequence_purged_val_idx, sequence_purged_test_idx, sequence_purge_audit = purge_sequence_context_splits(
        train_global_for_purge, val_global_for_purge, test_global_for_purge, CONFIG["sequence_purge_mode"]
    )
    global_to_local = np.full(len(metadata), -1, dtype=np.int64)
    global_to_local[active_idx] = np.arange(len(active_idx), dtype=np.int64)
    val_local = global_to_local[kept_val_global]
    test_local = global_to_local[kept_test_global]

train_idx = active_idx[train_local]
val_idx = active_idx[val_local]
test_idx = active_idx[test_local]
purged_val_idx = active_idx[purged_val_local]
purged_test_idx = active_idx[purged_test_local]
exact_variant_purged_val_idx = active_idx[exact_variant_purged_val_local]
exact_variant_purged_test_idx = active_idx[exact_variant_purged_test_local]

train_blocks = set(genome_block_groups[train_local])
val_blocks = set(genome_block_groups[val_local])
test_blocks = set(genome_block_groups[test_local])
split_audit = {
    "split_mode": CONFIG["split_mode"],
    "label_mode": CONFIG["label_mode"],
    "genome_block_size_bp": CONFIG["genome_block_size_bp"],
    "purge_distance_bp": CONFIG["purge_distance_bp"],
    "sequence_purge_mode": CONFIG["sequence_purge_mode"],
    "temporal_train_max_year": CONFIG["temporal_train_max_year"],
    "temporal_val_year": CONFIG["temporal_val_year"],
    "temporal_test_min_year": CONFIG["temporal_test_min_year"],
    "dropped_missing_last_evaluated_rows": int(dropped_missing_last_evaluated_rows),
    "selected_rows": int(len(active_idx)),
    "train_rows": int(len(train_idx)),
    "val_rows": int(len(val_idx)),
    "test_rows": int(len(test_idx)),
    "train_label_counts": label_count_dict(y_for_split, train_local),
    "val_label_counts": label_count_dict(y_for_split, val_local),
    "test_label_counts": label_count_dict(y_for_split, test_local),
    "train_prevalence": prevalence(y_for_split, train_local),
    "val_prevalence": prevalence(y_for_split, val_local),
    "test_prevalence": prevalence(y_for_split, test_local),
    "train_year_summary": year_summary(temporal_years, train_local),
    "val_year_summary": year_summary(temporal_years, val_local),
    "test_year_summary": year_summary(temporal_years, test_local),
    "train_block_count": int(len(train_blocks)),
    "val_block_count": int(len(val_blocks)),
    "test_block_count": int(len(test_blocks)),
    "genome_block_overlap_train_val": int(len(train_blocks & val_blocks)),
    "genome_block_overlap_train_test": int(len(train_blocks & test_blocks)),
    "genome_block_overlap_val_test": int(len(val_blocks & test_blocks)),
    "val_nearest_train_distance_bp": distance_summary(nearest_distance_to_reference(metadata_for_split, val_local, train_local)),
    "test_nearest_train_val_distance_bp": distance_summary(nearest_distance_to_reference(metadata_for_split, test_local, np.concatenate([train_local, val_local]))),
    "exact_variant_purge_audit": exact_variant_purge_audit,
    "coordinate_purge_audit": purge_audit,
    "sequence_purge_audit": sequence_purge_audit,
}
print(json.dumps(split_audit, indent=2))
assert split_audit["genome_block_overlap_train_val"] == 0 if CONFIG["split_mode"] in block_disjoint_modes else True
assert split_audit["genome_block_overlap_train_test"] == 0 if CONFIG["split_mode"] in block_disjoint_modes else True
assert split_audit["val_nearest_train_distance_bp"]["min"] >= CONFIG["purge_distance_bp"] if CONFIG["split_mode"] in coordinate_purge_modes else True
assert split_audit["test_nearest_train_val_distance_bp"]["min"] >= CONFIG["purge_distance_bp"] if CONFIG["split_mode"] in coordinate_purge_modes else True
if CONFIG["split_mode"] in {"time_only", "space_time_block"}:
    assert split_audit["train_year_summary"]["max"] <= CONFIG["temporal_train_max_year"]
    assert split_audit["val_year_summary"]["min"] == CONFIG["temporal_val_year"] == split_audit["val_year_summary"]["max"]
    assert split_audit["test_year_summary"]["min"] >= CONFIG["temporal_test_min_year"]


## 5. Dataset, reverse-complement và positional encoding

In [ ]:
def make_sinusoidal_position_encoding(length, dim):
    if dim == 0:
        return np.zeros((length, 0), dtype=np.float32)
    if dim % 2 != 0:
        raise ValueError("positional dim must be even")
    positions = np.arange(length, dtype=np.float32)[:, None]
    div_term = np.exp(np.arange(0, dim, 2, dtype=np.float32) * (-np.log(10000.0) / dim))
    pe = np.zeros((length, dim), dtype=np.float32)
    pe[:, 0::2] = np.sin(positions * div_term)
    pe[:, 1::2] = np.cos(positions * div_term)
    return pe


def make_relative_position_encoding(length, dim):
    if dim == 0:
        return np.zeros((length, 0), dtype=np.float32)
    if dim % 2 != 0:
        raise ValueError("positional dim must be even")
    center = (length - 1) / 2.0
    positions = (np.arange(length, dtype=np.float32) - center)[:, None]
    div_term = np.exp(np.arange(0, dim, 2, dtype=np.float32) * (-np.log(10000.0) / dim))
    pe = np.zeros((length, dim), dtype=np.float32)
    pe[:, 0::2] = np.sin(positions * div_term)
    pe[:, 1::2] = np.cos(positions * div_term)
    return pe


def make_position_encoding(kind, length, dim):
    if kind == "none":
        return np.zeros((length, 0), dtype=np.float32)
    if kind == "sinusoidal":
        return make_sinusoidal_position_encoding(length, dim)
    if kind == "relative":
        return make_relative_position_encoding(length, dim)
    raise ValueError(f"Unknown positional encoding: {kind}")


def reverse_complement_ref_alt_marker(sample):
    rc = sample[::-1].copy()
    rc[:, 0:4] = rc[:, [3, 2, 1, 0]]
    rc[:, 4:8] = rc[:, [7, 6, 5, 4]]
    return rc


class ClinVarSequenceDataset(Dataset):
    def __init__(self, x_array, y_array, indices, positional_encoding, random_reverse_complement=False, reverse_complement=False):
        self.x_array = x_array
        self.y_array = y_array
        self.indices = np.asarray(indices, dtype=np.int64)
        self.positional_encoding = positional_encoding.astype(np.float32, copy=False)
        self.random_reverse_complement = random_reverse_complement
        self.reverse_complement = reverse_complement
        if random_reverse_complement and reverse_complement:
            raise ValueError("Use either random or deterministic reverse-complement, not both.")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = int(self.indices[item])
        sample = self.x_array[idx].astype(np.float32, copy=True)
        if self.reverse_complement or (self.random_reverse_complement and random.random() < 0.5):
            sample = reverse_complement_ref_alt_marker(sample)
        sample = np.concatenate([sample, self.positional_encoding], axis=1)
        sample = np.ascontiguousarray(sample.T)
        return torch.from_numpy(sample), torch.tensor(np.float32(self.y_array[idx]))


def make_loader(indices, batch_size, sampler=None, shuffle=False, random_reverse_complement=False, reverse_complement=False):
    ds = ClinVarSequenceDataset(
        x, y, indices, positional_encoding,
        random_reverse_complement=random_reverse_complement,
        reverse_complement=reverse_complement,
    )
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        num_workers=CONFIG["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )


def make_weighted_sampler(indices):
    strategy = CONFIG["imbalance_strategy"]
    if strategy not in {"weighted_sampler", "legacy_sqrt_sampler_pos_weight"}:
        return None
    labels = y[indices].astype(int)
    counts = np.bincount(labels, minlength=2)
    if strategy == "weighted_sampler":
        weights_by_class = 1.0 / np.maximum(counts, 1)
    else:
        weights_by_class = 1.0 / np.sqrt(np.maximum(counts, 1))
    sample_weights = weights_by_class[labels]
    return WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)


def compute_pos_weight(indices):
    labels = y[indices].astype(int)
    counts = np.bincount(labels, minlength=2)
    strategy = CONFIG["imbalance_strategy"]
    if strategy == "pos_weight":
        return float(max(counts[0], 1) / max(counts[1], 1))
    if strategy == "legacy_sqrt_sampler_pos_weight":
        return float(np.sqrt(max(counts[0], 1) / max(counts[1], 1)))
    return 1.0

positional_encoding = make_position_encoding(CONFIG["positional_encoding"], x.shape[1], CONFIG["positional_dim"])
input_channels = int(x.shape[2] + positional_encoding.shape[1])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("input_channels:", input_channels)
print("positional shape:", positional_encoding.shape)

## 6. Model definitions

In [ ]:
class ClinVarCNN(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveMaxPool1d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1))

    def forward(self, inputs):
        return self.classifier(self.features(inputs)).squeeze(1)


class DilatedClinVarCNN(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        layers = [nn.Conv1d(in_channels, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ReLU()]
        for dilation in [1, 2, 4, 8, 16, 32, 64]:
            layers.extend([
                nn.Conv1d(64, 64, kernel_size=7, padding=3 * dilation, dilation=dilation),
                nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.05),
            ])
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Linear(64 * 3, 96), nn.ReLU(), nn.Dropout(0.25), nn.Linear(96, 1))

    def forward(self, inputs):
        features = self.features(inputs)
        center = features[:, :, features.shape[-1] // 2]
        global_max = torch.amax(features, dim=-1)
        global_mean = torch.mean(features, dim=-1)
        pooled = torch.cat([center, global_max, global_mean], dim=1)
        return self.classifier(pooled).squeeze(1)


class DilatedResidualBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.conv = nn.Conv1d(channels, channels, kernel_size=7, padding=3 * dilation, dilation=dilation)
        self.norm = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.05)

    def forward(self, inputs):
        out = self.dropout(self.relu(self.norm(self.conv(inputs))))
        return self.relu(inputs + out)


class DilatedResidualClinVarCNN(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_channels, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ReLU())
        self.blocks = nn.Sequential(*[DilatedResidualBlock(64, d) for d in [1, 2, 4, 8, 16, 32, 64]])
        self.classifier = nn.Sequential(nn.Linear(64 * 3, 96), nn.ReLU(), nn.Dropout(0.25), nn.Linear(96, 1))

    def forward(self, inputs):
        features = self.blocks(self.stem(inputs))
        center = features[:, :, features.shape[-1] // 2]
        global_max = torch.amax(features, dim=-1)
        global_mean = torch.mean(features, dim=-1)
        return self.classifier(torch.cat([center, global_max, global_mean], dim=1)).squeeze(1)


class WindowAttentionBlock1D(nn.Module):
    def __init__(self, channels=64, window_size=51, num_heads=4, dropout=0.10):
        super().__init__()
        self.window_size = window_size
        self.left_pad = window_size // 2
        self.norm_attn = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(channels, num_heads, dropout=dropout, batch_first=True)
        self.norm_ffn = nn.LayerNorm(channels)
        self.ffn = nn.Sequential(nn.Linear(channels, channels * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(channels * 2, channels), nn.Dropout(dropout))

    def forward(self, inputs):
        batch_size, channels, length = inputs.shape
        seq = inputs.transpose(1, 2)
        right_pad = (-length - self.left_pad) % self.window_size
        if self.left_pad or right_pad:
            seq = F.pad(seq, (0, 0, self.left_pad, right_pad))
        padded_length = seq.shape[1]
        windows = seq.reshape(batch_size, padded_length // self.window_size, self.window_size, channels).reshape(-1, self.window_size, channels)
        attn_input = self.norm_attn(windows)
        attn_output, _ = self.attn(attn_input, attn_input, attn_input, need_weights=False)
        windows = windows + attn_output
        windows = windows + self.ffn(self.norm_ffn(windows))
        seq = windows.reshape(batch_size, padded_length // self.window_size, self.window_size, channels).reshape(batch_size, padded_length, channels)
        seq = seq[:, self.left_pad:self.left_pad + length]
        return seq.transpose(1, 2)


class DilatedWindowAttentionClinVarCNN(DilatedClinVarCNN):
    def __init__(self, in_channels):
        super().__init__(in_channels)
        self.window_attention = WindowAttentionBlock1D(64, 51, 4, 0.10)

    def forward(self, inputs):
        features = self.window_attention(self.features(inputs))
        center = features[:, :, features.shape[-1] // 2]
        global_max = torch.amax(features, dim=-1)
        global_mean = torch.mean(features, dim=-1)
        pooled = torch.cat([center, global_max, global_mean], dim=1)
        return self.classifier(pooled).squeeze(1)


def make_model(architecture, in_channels):
    if architecture == "baseline":
        return ClinVarCNN(in_channels)
    if architecture == "dilated":
        return DilatedClinVarCNN(in_channels)
    if architecture == "dilated_residual":
        return DilatedResidualClinVarCNN(in_channels)
    if architecture == "dilated_window_attention":
        return DilatedWindowAttentionClinVarCNN(in_channels)
    raise ValueError(f"Unknown architecture: {architecture}")

print(make_model(CONFIG["architecture"], input_channels).__class__.__name__)

## 7. Metrics và train/eval helpers

In [ ]:
def safe_auc(y_true, y_score, kind):
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score) if kind == "roc" else average_precision_score(y_true, y_score))


def metrics_from_probs(y_true, y_prob, loss):
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "loss": float(loss),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "roc_auc": safe_auc(y_true, y_prob, "roc"),
        "pr_auc": safe_auc(y_true, y_prob, "pr"),
    }


def threshold_table(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    table = pd.DataFrame({"threshold": thresholds, "precision": precision[:-1], "recall": recall[:-1]})
    denom_f1 = table["precision"] + table["recall"]
    table["f1"] = np.where(denom_f1 > 0, 2 * table["precision"] * table["recall"] / denom_f1, 0.0)
    beta2 = 4.0
    denom_f2 = beta2 * table["precision"] + table["recall"]
    table["f2"] = np.where(denom_f2 > 0, (1 + beta2) * table["precision"] * table["recall"] / denom_f2, 0.0)
    return table


def metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_pathogenic": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall_pathogenic": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_pathogenic": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }


def make_scheduler(optimizer, total_steps):
    scheduler_name = CONFIG["scheduler"]
    if scheduler_name == "none" or total_steps <= 0:
        return None
    warmup_steps = max(0, min(int(round(CONFIG["warmup_epochs"] * len(train_loader))), total_steps - 1))
    min_lr_ratio = max(0.0, min(1.0, CONFIG["min_lr_ratio"]))
    if scheduler_name == "cosine":
        def lr_lambda(step):
            if warmup_steps > 0 and step < warmup_steps:
                return max((step + 1) / warmup_steps, min_lr_ratio)
            progress = min(1.0, max(0.0, (step - warmup_steps) / max(1, total_steps - warmup_steps)))
            cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
            return min_lr_ratio + (1.0 - min_lr_ratio) * cosine
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    if scheduler_name == "onecycle":
        pct_start = min(0.5, max(0.05, warmup_steps / max(total_steps, 1)))
        return torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=CONFIG["learning_rate"],
            total_steps=total_steps,
            pct_start=pct_start,
            div_factor=25.0,
            final_div_factor=max(1.0, 25.0 / max(min_lr_ratio, 1e-6)),
        )
    raise ValueError(f"Unknown scheduler: {scheduler_name}")


def run_epoch(model, loader, criterion, optimizer, label, scheduler=None):
    train = optimizer is not None
    model.train(train)
    losses, probs_all, labels_all = [], [], []
    last_lr = float(optimizer.param_groups[0]["lr"]) if train else float("nan")
    last_grad_norm = float("nan")
    for batch_x, batch_y in tqdm(loader, desc=label, leave=False):
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            if train:
                loss.backward()
                if CONFIG["grad_clip_norm"] and CONFIG["grad_clip_norm"] > 0:
                    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip_norm"])
                    last_grad_norm = float(grad_norm.detach().cpu())
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
                last_lr = float(optimizer.param_groups[0]["lr"])
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        labels = batch_y.detach().cpu().numpy()
        losses.append(float(loss.item()) * len(labels))
        probs_all.append(probs)
        labels_all.append(labels)
    y_true = np.concatenate(labels_all).astype(int)
    y_prob = np.concatenate(probs_all)
    metrics = metrics_from_probs(y_true, y_prob, np.sum(losses) / len(y_true))
    if train:
        metrics["lr"] = last_lr
        metrics["grad_norm"] = last_grad_norm
    return metrics, y_prob, y_true


def run_eval_epoch(model, loader, rc_loader, criterion, label, use_rc_tta):
    metrics, probs, labels = run_epoch(model, loader, criterion, None, label)
    if not use_rc_tta:
        return metrics, probs, labels
    rc_metrics, rc_probs, rc_labels = run_epoch(model, rc_loader, criterion, None, label + " rc")
    if not np.array_equal(labels, rc_labels):
        raise ValueError("RC loader label order mismatch")
    tta_probs = (probs + rc_probs) / 2.0
    tta_loss = (metrics["loss"] + rc_metrics["loss"]) / 2.0
    return metrics_from_probs(labels, tta_probs, tta_loss), tta_probs, labels

## 8. Train/evaluate inline

In [ ]:
def train_inline_model(save_outputs=True):
    output_paths = [paths["model"], paths["predictions"], paths["history"], paths["thresholds"], paths["val_thresholds"], paths["metrics"], paths["split_indices"]]
    if save_outputs and any(p.exists() for p in output_paths) and not FORCE_OVERWRITE:
        existing = "\n".join(str(p) for p in output_paths if p.exists())
        raise FileExistsError("Output exists. Set FORCE_OVERWRITE=True to overwrite:\n" + existing)

    global train_loader
    train_sampler = make_weighted_sampler(train_idx)
    train_loader = make_loader(
        train_idx,
        CONFIG["batch_size"],
        sampler=train_sampler,
        shuffle=train_sampler is None,
        random_reverse_complement=CONFIG["rc_augment"],
    )
    train_eval_loader = make_loader(train_idx, CONFIG["batch_size"])
    train_eval_rc_loader = make_loader(train_idx, CONFIG["batch_size"], reverse_complement=True) if CONFIG["rc_tta"] else None
    val_loader = make_loader(val_idx, CONFIG["batch_size"])
    val_rc_loader = make_loader(val_idx, CONFIG["batch_size"], reverse_complement=True) if CONFIG["rc_tta"] else None
    test_loader = make_loader(test_idx, CONFIG["batch_size"])
    test_rc_loader = make_loader(test_idx, CONFIG["batch_size"], reverse_complement=True) if CONFIG["rc_tta"] else None

    pos_weight_value = compute_pos_weight(train_idx)
    if pos_weight_value == 1.0:
        criterion = nn.BCEWithLogitsLoss()
    else:
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, dtype=torch.float32, device=device))
    model = make_model(CONFIG["architecture"], input_channels).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    scheduler = make_scheduler(optimizer, max(1, CONFIG["epochs"] * len(train_loader)))

    history = []
    best_val_pr_auc = -np.inf
    best_state = None
    start_time = time.time()

    for epoch in range(1, CONFIG["epochs"] + 1):
        print(f"Epoch {epoch}/{CONFIG['epochs']}")
        train_metrics, _, _ = run_epoch(model, train_loader, criterion, optimizer, f"train {epoch}", scheduler=scheduler)
        val_metrics, _, _ = run_eval_epoch(model, val_loader, val_rc_loader, criterion, f"val {epoch}", CONFIG["rc_tta"])
        row = {"phase": "main", "epoch": epoch, "scheduler": CONFIG["scheduler"], "warmup_epochs": CONFIG["warmup_epochs"], "grad_clip_norm": CONFIG["grad_clip_norm"]}
        row.update({f"train_{k}": v for k, v in train_metrics.items()})
        row.update({f"val_{k}": v for k, v in val_metrics.items()})
        history.append(row)
        print(row)
        if val_metrics["pr_auc"] > best_val_pr_auc:
            best_val_pr_auc = val_metrics["pr_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print("New best val_pr_auc:", f"{best_val_pr_auc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    if CONFIG["hard_negative_epochs"] > 0:
        print("Hard-negative mining on train set...")
        _, train_probs, train_labels_ordered = run_eval_epoch(model, train_eval_loader, train_eval_rc_loader, criterion, "predict train", CONFIG["rc_tta"])
        pos_idx = train_idx[train_labels_ordered == 1]
        hard_neg_idx = train_idx[(train_labels_ordered == 0) & (train_probs >= 0.50)]
        easy_neg_idx = train_idx[(train_labels_ordered == 0) & (train_probs < 0.50)]
        max_easy = int((len(pos_idx) + len(hard_neg_idx)) * 1.0)
        rng = np.random.default_rng(CONFIG["random_state"])
        if len(easy_neg_idx) > max_easy:
            easy_neg_idx = rng.choice(easy_neg_idx, size=max_easy, replace=False)
        hard_train_idx = np.concatenate([pos_idx, hard_neg_idx, easy_neg_idx])
        rng.shuffle(hard_train_idx)
        print("positive train rows:", f"{len(pos_idx):,}")
        print("hard negative rows:", f"{len(hard_neg_idx):,}")
        print("easy negative rows:", f"{len(easy_neg_idx):,}")
        print("fine-tune rows:", f"{len(hard_train_idx):,}")

        hard_sampler = make_weighted_sampler(hard_train_idx)
        hard_loader = make_loader(
            hard_train_idx,
            CONFIG["batch_size"],
            sampler=hard_sampler,
            shuffle=hard_sampler is None,
            random_reverse_complement=CONFIG["rc_augment"],
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["hard_negative_lr"], weight_decay=CONFIG["weight_decay"])
        for epoch in range(1, CONFIG["hard_negative_epochs"] + 1):
            print(f"Hard-negative epoch {epoch}/{CONFIG['hard_negative_epochs']}")
            train_metrics, _, _ = run_epoch(model, hard_loader, criterion, optimizer, f"hard train {epoch}")
            val_metrics, _, _ = run_eval_epoch(model, val_loader, val_rc_loader, criterion, f"hard val {epoch}", CONFIG["rc_tta"])
            row = {"phase": "hard_negative", "epoch": epoch, "scheduler": "none", "warmup_epochs": 0, "grad_clip_norm": CONFIG["grad_clip_norm"]}
            row.update({f"train_{k}": v for k, v in train_metrics.items()})
            row.update({f"val_{k}": v for k, v in val_metrics.items()})
            history.append(row)
            print(row)
            if val_metrics["pr_auc"] > best_val_pr_auc:
                best_val_pr_auc = val_metrics["pr_auc"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                print("New best val_pr_auc:", f"{best_val_pr_auc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    final_val_metrics, final_val_probs, final_val_labels = run_eval_epoch(model, val_loader, val_rc_loader, criterion, "final val", CONFIG["rc_tta"])
    test_metrics, test_probs, test_labels = run_eval_epoch(model, test_loader, test_rc_loader, criterion, "test", CONFIG["rc_tta"])
    test_pred = (test_probs >= 0.5).astype(int)
    print("Test metrics:", test_metrics)
    print(classification_report(test_labels, test_pred, target_names=["benign", "pathogenic"], zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(test_labels, test_pred))

    test_thresholds = threshold_table(test_labels, test_probs)
    val_thresholds = threshold_table(final_val_labels, final_val_probs)
    best_f1 = test_thresholds.loc[test_thresholds["f1"].idxmax()].to_dict()
    best_f2 = test_thresholds.loc[test_thresholds["f2"].idxmax()].to_dict()
    best_val_f1 = val_thresholds.loc[val_thresholds["f1"].idxmax()].to_dict()
    best_val_f2 = val_thresholds.loc[val_thresholds["f2"].idxmax()].to_dict()

    predictions = metadata.iloc[test_idx].copy()
    predictions["y_true"] = test_labels
    predictions["pred_proba_pathogenic"] = test_probs
    predictions["pred_label"] = test_pred
    predictions["pred_label_best_val_f1"] = (test_probs >= best_val_f1["threshold"]).astype(int)
    predictions["pred_label_best_val_f2"] = (test_probs >= best_val_f2["threshold"]).astype(int)

    val_prevalence = float(np.mean(final_val_labels.astype(np.float64)))
    test_prevalence = float(np.mean(test_labels.astype(np.float64)))
    val_pr_auc_lift = float(final_val_metrics["pr_auc"] / val_prevalence) if val_prevalence > 0 else float("nan")
    test_pr_auc_lift = float(test_metrics["pr_auc"] / test_prevalence) if test_prevalence > 0 else float("nan")

    metrics = {
        "model_name": SAFE_NAME,
        "length": CONFIG["length"],
        "dataset_tag": CONFIG["dataset_tag"],
        "dataset_suffix": DATASET_SUFFIX,
        "label_mode": CONFIG["label_mode"],
        "selected_rows": int(len(active_idx)),
        "input_channels": int(input_channels),
        "positional_encoding": CONFIG["positional_encoding"],
        "positional_dim": CONFIG["positional_dim"],
        "architecture": CONFIG["architecture"],
        "rc_augment": CONFIG["rc_augment"],
        "rc_tta": CONFIG["rc_tta"],
        "split": CONFIG["split_mode"],
        "purge_distance_bp": CONFIG["purge_distance_bp"],
        "genome_block_size_bp": CONFIG["genome_block_size_bp"],
        "sequence_purge_mode": CONFIG["sequence_purge_mode"],
        "split_audit": split_audit,
        "purge_audit": purge_audit,
        "sequence_purge_audit": sequence_purge_audit,
        "scheduler": CONFIG["scheduler"],
        "warmup_epochs": CONFIG["warmup_epochs"],
        "min_lr_ratio": CONFIG["min_lr_ratio"],
        "grad_clip_norm": CONFIG["grad_clip_norm"],
        "batch_size": CONFIG["batch_size"],
        "epochs": CONFIG["epochs"],
        "hard_negative_epochs": CONFIG["hard_negative_epochs"],
        "imbalance_strategy": CONFIG["imbalance_strategy"],
        "pos_weight": pos_weight_value,
        "train_rows": int(len(train_idx)),
        "val_rows": int(len(val_idx)),
        "test_rows": int(len(test_idx)),
        "val_prevalence": val_prevalence,
        "test_prevalence": test_prevalence,
        "val_pr_auc_lift": val_pr_auc_lift,
        "test_pr_auc_lift": test_pr_auc_lift,
        "final_val_metrics": final_val_metrics,
        "test_metrics": test_metrics,
        "test_precision_pathogenic_05": float(precision_score(test_labels, test_pred, zero_division=0)),
        "test_recall_pathogenic_05": float(recall_score(test_labels, test_pred, zero_division=0)),
        "test_f1_pathogenic_05": float(f1_score(test_labels, test_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(test_labels, test_pred).tolist(),
        "best_val_pr_auc": float(best_val_pr_auc),
        "best_f1_threshold": {k: float(v) for k, v in best_f1.items()},
        "best_f2_threshold": {k: float(v) for k, v in best_f2.items()},
        "best_val_f1_threshold": {k: float(v) for k, v in best_val_f1.items()},
        "best_val_f2_threshold": {k: float(v) for k, v in best_val_f2.items()},
        "test_at_best_val_f1_threshold": metrics_at_threshold(test_labels, test_probs, float(best_val_f1["threshold"])),
        "test_at_best_val_f2_threshold": metrics_at_threshold(test_labels, test_probs, float(best_val_f2["threshold"])),
        "runtime_minutes": float((time.time() - start_time) / 60.0),
    }

    if save_outputs:
        pd.DataFrame(history).to_parquet(paths["history"], index=False)
        predictions.to_parquet(paths["predictions"], index=False)
        test_thresholds.to_parquet(paths["thresholds"], index=False)
        val_thresholds.to_parquet(PROCESSED_DIR / f"{SAFE_NAME}_val_threshold_table_pytorch.parquet", index=False)
        np.savez_compressed(
            paths["split_indices"],
            train_idx=train_idx,
            val_idx=val_idx,
            test_idx=test_idx,
            purged_val_idx=purged_val_idx,
            purged_test_idx=purged_test_idx,
            exact_variant_purged_val_idx=exact_variant_purged_val_idx,
            exact_variant_purged_test_idx=exact_variant_purged_test_idx,
            sequence_purged_val_idx=sequence_purged_val_idx,
            sequence_purged_test_idx=sequence_purged_test_idx,
        )
        paths["metrics"].write_text(json.dumps(metrics, indent=2), encoding="utf-8")
        torch.save({"model_state_dict": model.state_dict(), "config": metrics}, paths["model"])
        print("Saved model:", paths["model"])
        print("Saved metrics:", paths["metrics"])

    return model, metrics, predictions, pd.DataFrame(history)

if RUN_TRAIN:
    model, metrics, predictions, history = train_inline_model(save_outputs=True)
else:
    print("RUN_TRAIN=False, skip inline training.")

## 9. Build dataset nếu thiếu

Artifact gốc `601` có thể không còn trong `processed`. Nếu muốn tái tạo dataset cũ, chạy command đầu. Nếu dùng pipeline đã fix accession GRCh38, chạy command thứ hai rồi đổi `CONFIG["dataset_tag"]` thành `"fixed_refseq"`.

In [20]:
legacy_build_cmd = [
    sys.executable,
    str(PROJECT_DIR / "scripts" / "build_cnn_dataset_1001.py"),
    "--flank", "300",
]

fixed_refseq_build_cmds = [
    [sys.executable, str(PROJECT_DIR / "scripts" / "build_clinvar_filtered_fixed_refseq.py")],
    [
        sys.executable,
        str(PROJECT_DIR / "scripts" / "build_cnn_dataset_1001.py"),
        "--flank", "300",
        "--input", str(PROCESSED_DIR / "clinvar_filtered_step8_fixed_refseq.parquet"),
        "--output-tag", "fixed_refseq",
    ],
]

print("Legacy 601 build:")
print(" ".join(legacy_build_cmd))
print("\nFixed RefSeq 601 build:")
for c in fixed_refseq_build_cmds:
    print(" ".join(c))

missing_dataset = [key for key in ["x", "y", "metadata"] if not paths[key].exists()]
print("\nMissing dataset artifacts:", missing_dataset)

Legacy 601 build:
/home/minato-one/.conda/envs/test-env/bin/python /mnt/MyData/Bioinformatics/Project/scripts/build_cnn_dataset_1001.py --flank 300

Fixed RefSeq 601 build:
/home/minato-one/.conda/envs/test-env/bin/python /mnt/MyData/Bioinformatics/Project/scripts/build_clinvar_filtered_fixed_refseq.py
/home/minato-one/.conda/envs/test-env/bin/python /mnt/MyData/Bioinformatics/Project/scripts/build_cnn_dataset_1001.py --flank 300 --input /mnt/MyData/Bioinformatics/Project/processed/clinvar_filtered_step8_fixed_refseq.parquet --output-tag fixed_refseq

Missing dataset artifacts: []


## 10. Metrics

Nếu metrics JSON tồn tại thì đọc từ file. Nếu không, cell dùng số lịch sử đã ghi lại cho đúng checkpoint cũ.

In [21]:
import pandas as pd

if paths["metrics"].exists():
    with open(paths["metrics"], "r", encoding="utf-8") as f:
        metrics = json.load(f)
    summary = {
        "model_name": metrics["model_name"],
        "length": metrics["length"],
        "dataset_tag": metrics.get("dataset_tag", ""),
        "split": metrics.get("split", "gene_group"),
        "architecture": metrics["architecture"],
        "input_channels": metrics["input_channels"],
        "positional_encoding": metrics.get("positional_encoding", "sinusoidal"),
        "positional_dim": metrics["positional_dim"],
        "rc_augment": metrics["rc_augment"],
        "rc_tta": metrics["rc_tta"],
        "best_val_pr_auc": metrics.get("best_val_pr_auc"),
        "test_roc_auc": metrics["test_metrics"]["roc_auc"],
        "test_pr_auc": metrics["test_metrics"]["pr_auc"],
        "test_accuracy": metrics["test_metrics"]["accuracy"],
        "test_f1_at_05": metrics["test_f1_pathogenic_05"],
        "test_precision_at_05": metrics["test_precision_pathogenic_05"],
        "test_recall_at_05": metrics["test_recall_pathogenic_05"],
        "best_val_f1_threshold": metrics["best_val_f1_threshold"]["threshold"],
        "test_f1_at_best_val_f1_threshold": metrics["test_at_best_val_f1_threshold"]["f1_pathogenic"],
    }
else:
    metrics = None
    summary = HISTORICAL_METRICS.copy()
    summary["source"] = "historical values; metrics JSON not found"

pd.Series(summary)

model_name                          cnn_gene_group_positional_encoding_601_fixed_r...
length                                                                            601
dataset_tag                                                              fixed_refseq
split                                                                    genome_block
architecture                                                                  dilated
input_channels                                                                     17
positional_encoding                                                        sinusoidal
positional_dim                                                                      8
rc_augment                                                                       True
rc_tta                                                                           True
best_val_pr_auc                                                              0.905226
test_roc_auc                                          

In [22]:
if paths["history"].exists():
    history = pd.read_parquet(paths["history"])
    display(history[[
        "phase", "epoch",
        "train_pr_auc", "val_pr_auc",
        "train_roc_auc", "val_roc_auc",
        "val_accuracy",
    ]])
else:
    print("Training history file not found:", paths["history"])

,phase,epoch,train_pr_auc,val_pr_auc,train_roc_auc,val_roc_auc,val_accuracy
0,main,1,0.768468,0.781876,0.889873,0.947525,0.894833
1,main,2,0.896260,0.846250,0.955550,0.966846,0.898059
2,main,3,0.924814,0.867836,0.969430,0.972404,0.916887
3,main,4,0.938168,0.882452,0.975442,0.975885,0.923197
4,main,5,0.947109,0.894257,0.979162,0.979049,0.928322
5,hard_negative,1,0.916449,0.902283,0.954897,0.980112,0.938401
6,hard_negative,2,0.924499,0.905226,0.959058,0.980860,0.939683


## 11. Load checkpoint để inference

Cell này dùng model/dataset/helper đã định nghĩa trong notebook, không import script train.

In [ ]:
required = ["x", "y", "metadata", "model"]
missing = [key for key in required if not paths[key].exists()]
if missing:
    print("Missing artifacts, skip model loading:", missing)
    for key in missing:
        print(f"- {key}:", paths[key])
    print("Nếu train full đang chạy, đợi cell train hoàn tất rồi chạy lại cell này.")
else:
    # Reload artifacts from disk so this cell works after restarting the kernel and running prior definition cells.
    x = np.load(paths["x"], mmap_mode="r")
    y = np.load(paths["y"], mmap_mode="r")
    metadata = pd.read_parquet(paths["metadata"])
    positional_encoding = make_position_encoding(CONFIG["positional_encoding"], x.shape[1], CONFIG["positional_dim"])
    input_channels = x.shape[2] + positional_encoding.shape[1]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = make_model(CONFIG["architecture"], input_channels).to(device)
    try:
        checkpoint = torch.load(paths["model"], map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(paths["model"], map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint)
    model.eval()
    print("device:", device)
    print("x shape:", x.shape, x.dtype)
    print("metadata shape:", metadata.shape)
    print("input_channels:", input_channels)
    print("checkpoint model:", checkpoint.get("config", {}).get("model_name", SAFE_NAME) if isinstance(checkpoint, dict) else SAFE_NAME)

In [ ]:
def predict_dataset_indices(indices, batch_size=512, threshold=None, use_rc_tta=True):
    if "model" not in globals():
        raise RuntimeError("Model is not loaded. Run the checkpoint cell after dataset/model artifacts exist.")
    indices = np.asarray(indices, dtype=np.int64)
    if threshold is None:
        if metrics is not None:
            threshold = metrics["best_val_f1_threshold"]["threshold"]
        else:
            threshold = HISTORICAL_METRICS["best_val_f1_threshold"]
    loader = make_loader(indices, batch_size=batch_size)
    rc_loader = make_loader(indices, batch_size=batch_size, reverse_complement=True) if use_rc_tta else None
    criterion = nn.BCEWithLogitsLoss()
    _, probs, labels = run_eval_epoch(model, loader, rc_loader, criterion, "predict", use_rc_tta)
    out = metadata.iloc[indices].copy()
    out["y_true"] = labels.astype(int)
    out["pred_proba_pathogenic"] = probs
    out["pred_label"] = (probs >= threshold).astype(int)
    out["threshold"] = float(threshold)
    return out

if "model" in globals():
    display(predict_dataset_indices(np.arange(8), batch_size=8)[[
        "GeneSymbol", "ClinicalSignificance", "Y", "y_true",
        "pred_proba_pathogenic", "pred_label", "threshold"
    ]])
else:
    print("Model not loaded; prediction demo skipped.")

## 12. Error analysis từ predictions đã lưu

Dùng khi đã có file predictions từ training script. Không cần chạy lại model.

In [25]:
if paths["predictions"].exists():
    predictions = pd.read_parquet(paths["predictions"])
    print(predictions.shape)
    display(predictions[[
        "GeneSymbol", "ClinicalSignificance", "Y", "y_true",
        "pred_proba_pathogenic", "pred_label"
    ]].head())
else:
    predictions = None
    print("Predictions file not found:", paths["predictions"])

(216351, 18)


,GeneSymbol,ClinicalSignificance,Y,y_true,pred_proba_pathogenic,pred_label
0,XPNPEP3,Pathogenic,1,1,0.989978,1
1,SDCCAG8,Pathogenic/Likely pathogenic,1,1,0.170186,0
2,SDCCAG8,Pathogenic,1,1,0.982020,1
3,TMEM127,Pathogenic,1,1,0.996223,1
4,TMEM127,Pathogenic/Likely pathogenic,1,1,0.999532,1


In [26]:
if predictions is not None:
    if metrics is not None:
        best_threshold = metrics["best_val_f1_threshold"]["threshold"]
    else:
        best_threshold = HISTORICAL_METRICS["best_val_f1_threshold"]

    error_view = predictions.assign(
        pred_label_best_val_f1=lambda df: (df["pred_proba_pathogenic"] >= best_threshold).astype(int),
        abs_error=lambda df: (df["pred_proba_pathogenic"] - df["y_true"]).abs(),
    )
    display(error_view.sort_values("abs_error", ascending=False)[[
        "GeneSymbol", "ClinicalSignificance", "ReferenceAlleleVCF", "AlternateAlleleVCF",
        "y_true", "pred_proba_pathogenic", "pred_label_best_val_f1", "abs_error"
    ]].head(20))
else:
    print("No predictions loaded; error analysis skipped.")

,GeneSymbol,ClinicalSignificance,ReferenceAlleleVCF,AlternateAlleleVCF,y_true,pred_proba_pathogenic,pred_label_best_val_f1,abs_error
142389,SIGLEC1,Benign,C,A,0,0.999957,1,0.999957
185554,DHCR24,Likely benign,G,T,0,0.999873,1,0.999873
174708,EYS,Likely pathogenic,C,G,1,0.000135,0,0.999865
204859,MYO1H,Likely benign,C,T,0,0.999842,1,0.999842
180058,TGFBR2,Likely benign,G,T,0,0.999800,1,0.999800
70585,TCIRG1,Pathogenic/Likely pathogenic,C,A,1,0.000346,0,0.999654
26043,TRPV4,Likely benign,C,T,0,0.999481,1,0.999481
49176,PCSK9,Likely benign,G,T,0,0.999479,1,0.999479
178489,MPZ,Likely pathogenic,C,A,1,0.000529,0,0.999471
139453,CHD7,Likely pathogenic,A,T,1,0.000539,0,0.999461
